# EDA: `Dataset` basics

Этот ноутбук специально сделан очень простым.

Цель:
- показать, как загрузить `Dataset`
- показать, какие DataFrame уже доступны из коробки
- показать несколько маленьких манипуляций, из которых удобно придумывать новые фичи

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.platform.core.dataset import Dataset


repo_root = Path.cwd().resolve()
if not (repo_root / "data").exists() and (repo_root.parent / "data").exists():
    repo_root = repo_root.parent

data_dir = repo_root / "data"
dataset = Dataset.load(data_dir)

data_dir

In [ ]:
frames = {
    "interactions_df": dataset.interactions_df,
    "targets_df": dataset.targets_df,
    "catalog_df": dataset.catalog_df,
    "authors_df": dataset.authors_df,
    "book_genres_df": dataset.book_genres_df,
    "genres_df": dataset.genres_df,
    "users_df": dataset.users_df,
    "seen_positive_df": dataset.seen_positive_df,
}

summary = pd.DataFrame(
    {
        "name": list(frames.keys()),
        "rows": [len(frame) for frame in frames.values()],
        "columns": [list(frame.columns) for frame in frames.values()],
    }
)

summary

In [ ]:
for name, frame in frames.items():
    print(f"\n{name}")
    display(frame.head(3))

## Очень простые примеры манипуляций

Ниже не «готовые фичи», а маленькие заготовки, из которых удобно начинать EDA и feature engineering.

In [ ]:
positives = dataset.interactions_df[
    dataset.interactions_df["event_type"].isin([1, 2])
].copy()

print("positive rows:", len(positives))
print("target users:", dataset.targets_df["user_id"].nunique())
print("unique seen-positive pairs:", len(dataset.seen_positive_df))
print("time range:", positives["event_ts"].min(), "->", positives["event_ts"].max())

positives.head()

In [ ]:
catalog_with_authors = dataset.catalog_df.merge(
    dataset.authors_df,
    on="author_id",
    how="left",
)

catalog_with_authors[["edition_id", "book_id", "author_id", "author_name"]].head()

In [ ]:
catalog_with_genres = (
    dataset.catalog_df[["edition_id", "book_id", "author_id"]]
    .merge(dataset.book_genres_df, on="book_id", how="left")
    .merge(dataset.genres_df, on="genre_id", how="left")
)

catalog_with_genres.head()

In [ ]:
user_author_profile = (
    positives[["user_id", "edition_id"]]
    .merge(dataset.catalog_df[["edition_id", "author_id"]], on="edition_id", how="left")
    .groupby(["user_id", "author_id"], as_index=False)["edition_id"]
    .count()
    .rename(columns={"edition_id": "positive_events"})
    .sort_values(["user_id", "positive_events"], ascending=[True, False])
)

user_author_profile.head(10)

In [ ]:
user_genre_profile = (
    positives[["user_id", "edition_id"]]
    .merge(dataset.catalog_df[["edition_id", "book_id"]], on="edition_id", how="left")
    .merge(dataset.book_genres_df, on="book_id", how="left")
    .groupby(["user_id", "genre_id"], as_index=False)["edition_id"]
    .count()
    .rename(columns={"edition_id": "positive_events"})
    .merge(dataset.genres_df, on="genre_id", how="left")
    .sort_values(["user_id", "positive_events"], ascending=[True, False])
)

user_genre_profile.head(10)

## Что из этого удобно брать в новые фичи

Обычно самые простые кандидаты на новые фичи начинаются с таких комбинаций:

- `dataset.interactions_df` + `dataset.catalog_df` — признаки по автору, году, языку, издателю
- `dataset.interactions_df` + `dataset.book_genres_df` + `dataset.genres_df` — жанровые предпочтения
- `dataset.catalog_df` + `dataset.authors_df` — авторские словари и агрегации по авторам
- `dataset.targets_df` + любые user-level агрегации — фичи только для target users
- `dataset.seen_positive_df` — фильтрация уже виденных позитивных пар

Если хочется продолжить, самый естественный следующий шаг — открыть `src/competition/features.py` и попробовать повторить одну из этих агрегаций в виде новой feature table.